In [30]:
import os
import numpy as np
import rasterio
from tqdm import tqdm
import matplotlib.pyplot as plt

In [31]:
# ===============================
# 1. PREPROCESSING FUNCTIONS
# ===============================

def extract_features_from_sar(img):
    vv = img[0].astype(np.float32)
    vh = img[1].astype(np.float32)
    ratio = vv / (vh + 1e-8)
    return np.stack([vv, vh, ratio], axis=0)

def global_normalize(features):
    mean = np.mean(features)
    std = np.std(features)
    return (features - mean) / (std + 1e-8)

In [32]:
MAIN_DIR = os.getcwd()

# Adjust these to your actual Zenodo Part I paths
BASE_INPUT = "Sentinel-1 SAR Oil spill image train. Part I"
IMG_DIR  = os.path.join(MAIN_DIR, BASE_INPUT, "01_Train_Val_Oil_Spill_images/Oil")
MASK_DIR = os.path.join(MAIN_DIR, BASE_INPUT, "01_Train_Val_Oil_Spill_mask/Mask_oil")

# Your requested output structure
SAVE_IMG  = os.path.join(MAIN_DIR, "oil_images_preprocessed", "oil_images")
SAVE_MASK = os.path.join(MAIN_DIR, "oil_images_preprocessed", "oil_masks")

os.makedirs(SAVE_IMG, exist_ok=True)
os.makedirs(SAVE_MASK, exist_ok=True)

print(f"Input Images: {IMG_DIR}")
print(f"Saving Patches to: {os.path.join(MAIN_DIR, 'oil_images_preprocessed')}")

Input Images: /home/nebu_placid/main_project/pre_process/Sentinel-1 SAR Oil spill image train. Part I/01_Train_Val_Oil_Spill_images/Oil
Saving Patches to: /home/nebu_placid/main_project/pre_process/oil_images_preprocessed


In [33]:
# ===============================
# 3. MAIN PROCESSING LOOP (OIL)
# ===============================

idx = 0  
TARGET_PATCHES = 6400  # 64 * 100 images
MIN_OIL_PIXELS = 1500   #  ENFORCED: Remove small patches (Adjust as needed)
processed_count = 0

all_files = sorted([f for f in os.listdir(IMG_DIR) if f.endswith('.tif')])

print(f"Targeting: {TARGET_PATCHES} high-quality oil patches...")
pbar = tqdm(total=TARGET_PATCHES, desc="Filtering Oil Patches")

for fname in all_files:
    if idx >= TARGET_PATCHES:
        break

    img_path = os.path.join(IMG_DIR, fname)
    mask_path = os.path.join(MASK_DIR, fname)

    if not os.path.exists(mask_path):
        continue

    # Read SAR and Mask
    with rasterio.open(img_path) as src:
        img = src.read()  
    with rasterio.open(mask_path) as src:
        mask = (src.read(1) > 0).astype(np.uint8)

    # Preprocessing (CPU Logic)
    features = extract_features_from_sar(img)
    features = global_normalize(features)

    # Patching with ENFORCED removal of small oil clusters
    for i in range(0, 2048, 256):
        for j in range(0, 2048, 256):
            if idx >= TARGET_PATCHES:
                break
                
            p_mask = mask[i:i+256, j:j+256]
            oil_pixel_count = np.sum(p_mask)

            # 🔑 THE ENFORCED CONSTRAINT
            # Only save if oil is significant (greater than MIN_OIL_PIXELS)
            if oil_pixel_count >= MIN_OIL_PIXELS:
                p_img = features[:, i:i+256, j:j+256]
                
                # Save as /oil_images_preprocessed/oil_images/00001.npy
                np.save(os.path.join(SAVE_IMG, f"{idx:05d}.npy"), p_img)
                np.save(os.path.join(SAVE_MASK, f"{idx:05d}.npy"), p_mask)
                
                idx += 1
                pbar.update(1)

    processed_count += 1

pbar.close()
print(f"\nPreprocessing completed!")
print(f"Total .tif images searched: {processed_count}")
print(f"Total high-quality oil patches saved: {idx}")

Targeting: 6400 high-quality oil patches...


Filtering Oil Patches: 100%|████████████████████████████████████████████████████████| 6400/6400 [03:08<00:00, 34.02it/s]


Preprocessing completed!
Total .tif images searched: 634
Total high-quality oil patches saved: 6400


In [ ]:
# ===============================
# 4. VERIFICATION & VISUALIZATION
# ===============================

def visualize_results(save_img_path, save_mask_path, n=4):
    # Get sorted list of files (00000.npy, 00001.npy, etc.)
    img_files = sorted([f for f in os.listdir(save_img_path) if f.endswith('.npy')])
    
    if not img_files:
        print(f"No .npy files found in {save_img_path}")
        return

    # Limit to 'n' or the total number of files available
    n = min(n, len(img_files))
    
    fig, axes = plt.subplots(n, 4, figsize=(16, 4 * n))
    titles = ["VV Channel", "VH Channel", "Ratio (VV/VH)", "Mask (Label)"]

    for i in range(n):
        fname = img_files[i]
        
        # Load the matching pair using the EXACT same filename
        img_patch = np.load(os.path.join(save_img_path, fname))
        mask_patch = np.load(os.path.join(save_mask_path, fname))

        for j in range(3):
            # Rescale the globally normalized data to 0-1 just for visualization
            data = img_patch[j]
            data_vis = (data - data.min()) / (data.max() - data.min() + 1e-8)
            
            # Select axis (handle case if n=1)
            ax = axes[i, j] if n > 1 else axes[j]
            ax.imshow(data_vis, cmap='gray')
            ax.set_title(f"{titles[j]}\n{fname}")
            ax.axis('off')

        # Display the mask
        ax_mask = axes[i, 3] if n > 1 else axes[3]
        ax_mask.imshow(mask_patch, cmap='viridis')
        ax_mask.set_title(f"Ground Truth\n{fname}")
        ax_mask.axis('off')

    plt.tight_layout()
    plt.show()

# To visualize the Oil patches you just created:
visualize_results(SAVE_IMG, SAVE_MASK, n=100)